In [72]:
%reset -f

# Filter out Climate-related Text

In [73]:
%%writefile climate_sift1.py

import argparse
import csv
import html
import json
import random
import re
import sys
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple


# ----------------------
# thresholds & keywords
# ----------------------
MIN_LINES = 1            
MIN_WORDS = 10           
MIN_UNIQUE_WORDS = 6     
MIN_LETTER_RATIO = 0.60
REQUIRE_PERIOD = True

KEYWORDS = {
    "air quality", "water quality", "bushfire", "carbon", "ch4",
    "climate", "climate-related", "co2", "coal",
    "decarbonization", "decarbonisation", "deforestation", "drought",
    "emission", "energy consumption", "energy efficiency", "energy efficient",
    "energy transition", "environmental", "esg", "footprint",
    "fossil", "ghg", "global warming", "greenhouse",
    "heat wave", "hurricane", "land use", "litigation risk",
    "low-carbon", "methane", "n2o", "natural hazard",
    "nitrous oxide", "o3", "ozone", "paris agreement",
    "physical risk", "renewable", "rural fire", "sea level",
    "social responsibility", "solar energy", "sustainable", "sustainability",
    "tcfd", "temperature rise", "transition risk", "tropical cyclone",
    "tropical storm", "typhoon", "weather", "wildfire", "wildland fire", "wind energy",
    "hazardous substances", "pollutants", "contamination", "scope 3", "scope 2", "scope 1",
    "waste", "cleanup", "disposal"
}

# ----------------------
# utils and regex
# ----------------------
SECTION_HDR_RE = re.compile(
    r"^(?P<hashes>#{10})\s*\d+\.\s*(?P<title>.+?)\s*(?P=hashes)\s*$",
    re.MULTILINE
)
WORD_RE = re.compile(r"\b[\w']+\b", re.UNICODE)  
ALPHA_RE = re.compile(r"[A-Za-z]") 
KW_GROUPMAP: Dict[int, str] = {}  

def log(msg: str):
    print(msg, file=sys.stderr)


def normalize_text(s: str) -> str:
    return (s.replace("–", "-").replace("—", "-")
             .replace("’", "'").replace("‘", "'")
             .replace("“", '"').replace("”", '"'))

# detect sections
def split_sections(s: str) -> List[Tuple[str, str]]:
    out, last_end, last_title = [], 0, None
    found = False
    for m in SECTION_HDR_RE.finditer(s):
        found = True
        if last_title is not None:
            out.append((last_title, s[last_end:m.start()].strip("\n")))
        last_title = m.group("title").strip()
        last_end = m.end()
    
    if not found:
        raise ValueError("split_sections: do not find any titles")

    out.append((last_title, s[last_end:].strip("\n")))
    return out

# character statistics

def paragraph_metrics(p: str) -> Dict[str, object]:
    lines = [ln for ln in p.splitlines() if ln.strip()]
    tokens = WORD_RE.findall(p)
    words = [w.lower() for w in tokens]
    num_words = len(words)
    num_unique = len(set(words))
    no_ws = re.sub(r"\s+", "", p)
    total_chars = len(no_ws)
    alpha_chars = len(ALPHA_RE.findall(p))
    return {
        "num_lines": len(lines),
        "num_words": num_words,
        "num_unique_words": num_unique,
        "letter_ratio": (alpha_chars / total_chars) if total_chars else 0.0,
        "has_period": bool(re.search(r'[.!?]', p)),
    }


def meets_thresholds(m: Dict[str, object]) -> bool:
    if m["num_lines"] < MIN_LINES: return False
    if m["num_words"] < MIN_WORDS: return False
    if m["num_unique_words"] < MIN_UNIQUE_WORDS: return False
    if m["letter_ratio"] < MIN_LETTER_RATIO: return False
    if REQUIRE_PERIOD and not m["has_period"]: return False
    return True

def _strip_leading_marker_and_title(p: str) -> str:
    return p.lstrip()

# ----------------------
# split sentences
# ----------------------

def sentence_candidates(section_text: str) -> List[str]:
    if not section_text:
        return []

    enum_pat = r'(?m)(?=^\s*(?:\(\d{1,3}\)|\d{1,3}[.)]|[A-Za-z][.)]|[IVXLCDM]{1,6}[.)]|[•\-\*])\s+)'
    s = re.sub(enum_pat, ' <SPLIT> ', section_text)
    s = re.sub(r'(?:\r?\n\s*){2,}', ' <SPLIT> ', s)

    s = re.sub(r'(\w)-\s*\n\s*(\w)', r'\1\2', s)
    s = re.sub(r"\s*\n+\s*", " ", s).strip()  

    DOT = "\u2024"  # one-dot leader
    ABBR_PAT = re.compile(
        r'\b(?:Mr|Mrs|Ms|Dr|Prof|Sr|Jr|St|Mt|Gen|Col|Capt|Sgt|vs|No|Nos|pp|Vol|Eq|Fig|Dept|Inc|Ltd|Co|Corp|LLC|Jan|Feb|Mar|Apr|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec|U\.S|U\.K|e\.g|i\.e|etc)\.',
        re.IGNORECASE
    )
    s = ABBR_PAT.sub(lambda m: m.group(0).replace('.', DOT), s)
    s = re.sub(r'(?<=\d)\.(?=\d)', DOT, s)

    parts = re.split(r'([.!?]+[\'"\)\]]*)\s+', s)


    tmp = []
    for i in range(0, len(parts), 2):
        piece = (parts[i].strip() + (parts[i+1] if i+1 < len(parts) else '')).strip()
        if piece:
            tmp.append(piece.replace(DOT, '.'))

    final = []
    for t in tmp:
        segs = [x.strip() for x in t.split('<SPLIT>') if x.strip()]
        final.extend(segs)

    return final


# ----------------------
# match keywords
# ----------------------
def _keyword_to_pattern(k: str) -> str:
    parts = re.split(r"[\s\-]+", k.lower().strip())
    mid = r"[-\s_]*"
    core = mid.join(re.escape(p) for p in parts if p)
    return rf"(?<![A-Za-z0-9]){core}(?![A-Za-z0-9])"

def build_keywords_regex(keywords: Iterable[str]) -> re.Pattern:
    global KW_GROUPMAP
    KW_GROUPMAP = {}
    uniq_sorted = sorted({kw.lower().strip() for kw in keywords},
                         key=lambda s: (-len(s), s))
    if not uniq_sorted:
        return re.compile(r"(?!)") 
    patterns = []
    for i, kw in enumerate(uniq_sorted, start=1):
        patterns.append(f"({_keyword_to_pattern(kw)})")
        KW_GROUPMAP[i] = kw
    return re.compile("|".join(patterns), re.IGNORECASE)

KW_REGEX = build_keywords_regex(KEYWORDS)

def find_keywords(text: str) -> List[str]:
    hits = set()
    for m in KW_REGEX.finditer(text):
        gi = m.lastindex 
        if gi and gi in KW_GROUPMAP:
            hits.add(KW_GROUPMAP[gi])
    return sorted(hits)


def format_cik_for_excel(cik: str, mode: str = "none") -> str:
    s = str(cik)
    if mode == "apostrophe":
        return s if s.startswith("'") else "'" + s
    return s


# ----------------------
# data model
# ----------------------
@dataclass
class ParaRow:
    cik: str
    section_index: int
    section: str
    para_idx: int
    num_lines: int
    num_words: int
    num_unique_words: int
    letter_ratio: float
    has_period: bool
    is_climate: bool
    matched_keywords: str
    paragraph: str
    preview: str
    source_file: str


# ----------------------
# processing
# ----------------------
def process_one_file(in_path: Path, company_name: Optional[str] = None) -> List[ParaRow]:
    cik = in_path.stem
    txt = normalize_text(in_path.read_text(encoding="utf-8", errors="ignore"))
    sections = split_sections(txt)

    rows: List[ParaRow] = []
    for s_idx, (title, s_text) in enumerate(sections, start=1):
        for p_idx, p in enumerate(sentence_candidates(s_text), start=1):
            p = _strip_leading_marker_and_title(p)
            m = paragraph_metrics(p)
            if not meets_thresholds(m): 
                continue
            hits = find_keywords(p.lower())
            # only hitting 'proposal' with at least one other keyword counts as a match
            is_hit = any(h != "disposal" for h in hits)  

            preview = re.sub(r"\s+", " ", p).strip()
            if len(preview) > 400: 
                preview = preview[:400] + "…"
            rows.append(ParaRow(
                cik=cik,
                section_index=s_idx,
                section=title,
                para_idx=p_idx,
                num_lines=m["num_lines"],
                num_words=m["num_words"],
                num_unique_words=m["num_unique_words"],
                letter_ratio=round(m["letter_ratio"],4),
                has_period=m["has_period"],
                is_climate=is_hit,
                matched_keywords=";".join(hits),
                paragraph=p,
                preview=preview,
                source_file=str(in_path)
            ))
    return rows

# ----------------------
# exports
# ----------------------
def write_csv(path: Path, rows: List[ParaRow], excel_mode: str = "none"):
    path.parent.mkdir(parents=True, exist_ok=True)
    from copy import deepcopy
    with path.open("w", newline="", encoding="utf-8") as f:
        fieldnames = list(asdict(rows[0]).keys()) if rows else [
            "cik","section_index","section","para_idx","num_lines","num_words",
            "num_unique_words","letter_ratio","has_period","is_climate","matched_keywords",
            "paragraph","preview","source_file"
        ]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for r in rows:
            row = deepcopy(asdict(r))
            row["cik"] = format_cik_for_excel(r.cik, excel_mode)
            writer.writerow(row)
    if excel_mode == "xlsx" and rows:
        write_rows_xlsx(path.with_suffix(".xlsx"), rows)

def write_rows_xlsx(xlsx_path: Path, rows: List[ParaRow]):
    try:
        import pandas as pd
    except Exception as e:
        log(f"[WARN] not find pandas, skip {xlsx_path}：{e}")
        return
    import pandas as pd
    df = pd.DataFrame([asdict(r) for r in rows])
    df["cik"] = df["cik"].astype(str)
    xlsx_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_excel(xlsx_path, index=False)

def write_corpus_tsv(out_dir: Path, rows: List[ParaRow], excel_mode: str = "none"):
    corpus_dir = out_dir / "corpus"
    corpus_dir.mkdir(parents=True, exist_ok=True)
    tsv_path = corpus_dir / "corpus_climate.txt"
    with tsv_path.open("w", encoding="utf-8", newline="") as f:
        f.write("cik\tsection\tsection_idx\tpara_idx\tkeywords\ttext\tsource_file\n")
        for r in rows:
            if not r.is_climate:
                continue
            cik_out = format_cik_for_excel(r.cik, excel_mode)
            text_1line = re.sub(r"\s+", " ", r.paragraph.strip())
            sec = r.section.replace("\t", " ").strip()
            f.write(
                f"{cik_out}\t{sec}\t{r.section_index}\t{r.para_idx}\t"
                f"{r.matched_keywords}\t{text_1line}\t{r.source_file}\n"
            )
    
    (corpus_dir / "corpus_climate_text_only.txt").write_text(
        "\n\n".join([r.paragraph.strip() for r in rows if r.is_climate]),
        encoding="utf-8"
    )
    if excel_mode == "xlsx":
        write_corpus_xlsx(corpus_dir / "corpus_climate.xlsx", rows)


def write_corpus_xlsx(xlsx_path: Path, rows: List[ParaRow]):
    try:
        import pandas as pd
    except Exception as e:
        log(f"[WARN] not find pandas, skip {xlsx_path}: {e}")
        return
    data = []
    for r in rows:
        if not r.is_climate:
            continue
        data.append({
            "cik": str(r.cik),
            "section": r.section,
            "section_idx": r.section_index,
            "para_idx": r.para_idx,
            "keywords": r.matched_keywords,
            "text": re.sub(r"\s+", " ", r.paragraph.strip()),
            "source_file": r.source_file,
        })
    if not data:
        return
    df = pd.DataFrame(data)
    xlsx_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_excel(xlsx_path, index=False)

def _build_split_plan_from_tasks(tasks: List[dict], ratios=(0.70, 0.15, 0.15), seed: int = 13) -> Dict[str, List[str]]:
    rng = random.Random(seed)
    ciks = sorted({
        str((t.get("data") or {}).get("cik") or "").strip()
        for t in tasks if (t.get("data") or {}).get("cik")
    })
    rng.shuffle(ciks)
    n = len(ciks)
    n_tr = int(n * ratios[0])
    n_va = int(n * ratios[1])
    plan = {
        "train": ciks[:n_tr],
        "val":   ciks[n_tr:n_tr + n_va],
        "test":  ciks[n_tr + n_va:],
    }
    return plan

def _apply_split_meta(tasks: List[dict], plan: Dict[str, List[str]]) -> List[dict]:
    tr, va, te = set(plan.get("train", [])), set(plan.get("val", [])), set(plan.get("test", []))
    for t in tasks:
        d = t.setdefault("data", {})
        cik = str(d.get("cik", "")).strip()
        if cik in va:
            split = "val"
        elif cik in te:
            split = "test"
        else:
            split = "train" 
        t.setdefault("meta", {})["split"] = split
    return tasks


def build_annot_candidates(
    rows: List[ParaRow],
    total: int = 1000,           
    seed: int = 13,
    per_cik_cap: int = 5      
) -> List[dict]:
    rng = random.Random(seed)
    if not rows:
        return []

    pos = [r for r in rows if r.is_climate]
    neg = [r for r in rows if not r.is_climate]

    tgt_pos = min(total // 2, len(pos))
    tgt_neg = min(total - tgt_pos, len(neg))
    if tgt_pos + tgt_neg < total:
        more_pos = min(total - (tgt_pos + tgt_neg), len(pos) - tgt_pos)
        tgt_pos += more_pos
        more_neg = min(total - (tgt_pos + tgt_neg), len(neg) - tgt_neg)
        tgt_neg += more_neg

    def pick_class(bucket_src: List[ParaRow], k: int) -> List[ParaRow]:
        by_cik: Dict[str, List[ParaRow]] = {}
        for r in bucket_src:
            by_cik.setdefault(r.cik, []).append(r)
        for v in by_cik.values():
            rng.shuffle(v)

        keys = list(by_cik.keys())
        rng.shuffle(keys)
        picked: List[ParaRow] = []
        idx_map = {k: 0 for k in keys}
        cap_map = {k: min(per_cik_cap, len(by_cik[k])) for k in keys}

        while len(picked) < k and keys:
            next_keys = []
            for cik in keys:
                i = idx_map[cik]
                cap = cap_map[cik]
                if i < cap:
                    picked.append(by_cik[cik][i])
                    idx_map[cik] = i + 1
                    if idx_map[cik] < cap:
                        next_keys.append(cik)
                if len(picked) >= k:
                    break
            keys = next_keys

        if len(picked) < k:
            remain = [r for lst in by_cik.values() for r in lst]
            rng.shuffle(remain)
            seen = {(r.cik, r.section_index, r.para_idx, r.source_file) for r in picked}
            for r in remain:
                key = (r.cik, r.section_index, r.para_idx, r.source_file)
                if key not in seen:
                    picked.append(r); seen.add(key)
                    if len(picked) >= k:
                        break
        return picked

    pos_pick = pick_class(pos, tgt_pos)
    neg_pick = pick_class(neg, tgt_neg)

    picked = pos_pick + neg_pick
    rng.shuffle(picked)

    tasks: List[dict] = []
    for r in picked:
        tasks.append({
            "data": {
                "text": r.paragraph,
                "cik": str(r.cik),
                "section": r.section,
                "para_idx": r.para_idx,
                "is_kw_hit": int(r.is_climate),
                "source_file": r.source_file
            }
        })
    return tasks

def _write_ann_by_split(ann_dir: Path, tasks: List[Dict], pretty: bool = True):
    buckets = {"train": [], "val": [], "test": []}
    for t in tasks:
        split = ((t.get("meta") or {}).get("split") or "train").lower()
        if split not in buckets:
            split = "train"
        buckets[split].append(t)

    for name, arr in buckets.items():
        out = ann_dir / f"labelstudio_tasks_{name}.json"
        with out.open("w", encoding="utf-8") as f:
            if pretty:
                json.dump(arr, f, ensure_ascii=False, indent=2)
            else:
                json.dump(arr, f, ensure_ascii=False, separators=(",", ":"))


def write_ann_candidates(
    out_dir: Path,
    tasks: List[Dict],
    filename: str = "labelstudio_tasks.json",
    pretty: bool = True,
    seed_for_split: int = 13
) -> Path:

    ann_dir = out_dir / "ann"; ann_dir.mkdir(parents=True, exist_ok=True)

    split_plan = _build_split_plan_from_tasks(tasks, ratios=(0.70, 0.15, 0.15), seed=seed_for_split)
    tasks_tagged = _apply_split_meta(tasks, split_plan)

    out_path = ann_dir / filename
    with out_path.open("w", encoding="utf-8") as f:
        if pretty:
            json.dump(tasks_tagged, f, ensure_ascii=False, indent=2)
        else:
            json.dump(tasks_tagged, f, ensure_ascii=False, separators=(",", ":"))

    _write_ann_by_split(ann_dir, tasks_tagged, pretty=pretty)

    with (ann_dir / "split_plan_cik.json").open("w", encoding="utf-8") as f:
        json.dump(split_plan, f, ensure_ascii=False, indent=2)

    return out_path



# ----------------------
# extract
# ----------------------
def _collect_files(in_file: Optional[str], in_dir: Optional[str]) -> List[Path]:
    files: List[Path] = []
    if in_file:
        p = Path(in_file)
        if p.is_file():
            files.append(p)
        else:
            raise FileNotFoundError(f"--in_file not found: {in_file}")
    if in_dir:
        d = Path(in_dir)
        if not d.exists():
            raise FileNotFoundError(f"--in_dir not found: {in_dir}")
        for p in sorted(d.rglob("*.txt")):
            if p.is_file():
                files.append(p)
    if not files:
        raise ValueError("No input files. Provide --in_file or --in_dir with *.txt files.")
    return files


def cmd_extract(args):
    out_dir = Path(args.out); out_dir.mkdir(parents=True, exist_ok=True)
    files = _collect_files(args.in_file, args.in_dir)

    # only process the former N files
    #if getattr(args, "head", 0):
       # files = files[:args.head]

    all_rows: List[ParaRow] = []
    for p in files:
        all_rows.extend(process_one_file(p))

    if all_rows:
        write_csv(out_dir / "csv" / "paragraphs_pass_thresholds.csv",
                all_rows, excel_mode=args.excel_mode)
        write_csv(out_dir / "csv" / "climate_related_paragraphs.csv",
                [r for r in all_rows if r.is_climate],
                excel_mode=args.excel_mode)
        write_corpus_tsv(out_dir, all_rows, excel_mode=args.excel_mode)

    # label
    tasks = build_annot_candidates(
        all_rows,
        total=args.ann_total,
        seed=args.seed,
        per_cik_cap=args.per_cik_cap
    )
    if tasks:
        write_ann_candidates(out_dir, tasks, seed_for_split=args.seed)

    print(f"[DONE] files={len(files)}, paras={len(all_rows)}, climate={sum(r.is_climate for r in all_rows)}")


# ----------------------
# main
# ----------------------
def main():
    ap = argparse.ArgumentParser(
        description="Climate paragraph pipeline (CIK-aware)",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter
    )
    sub = ap.add_subparsers(dest="cmd", required=True)

    sp = sub.add_parser(
        "extract",
        help="Split / filter / save & build annotation candidates (filenames = CIK)"
    )
    sp.add_argument("--in_file", help="Single input file (filename = 10-digit CIK.txt)")
    sp.add_argument("--in_dir", help="Input directory (recursively search *.txt; filename = 10-digit CIK)")
    #sp.add_argument("--head", type=int, default=0,
               # help="Only process the first N files (after sorting)")  # for trial
    sp.add_argument("--out", required=True, help="Output directory")
    sp.add_argument("--ann_total", type=int, default=1000,
                    help="Total annotation candidates with 1:1 balance (pos/neg). Default 1000.")
    sp.add_argument("--per_cik_cap", type=int, default=5,
                    help="Max samples per class per CIK to avoid concentration. Default 5.")
    sp.add_argument("--seed", type=int, default=13, help="Random seed")
    sp.add_argument("--excel_mode", choices=["none", "apostrophe", "xlsx"], default="none",
                    help="How to write CIK in CSV: none=as-is; apostrophe=prefix ' to keep leading zeros in Excel; xlsx=also write a .xlsx copy")
    sp.set_defaults(func=cmd_extract)

    args = ap.parse_args()
    args.func(args)


if __name__ == "__main__":
    main()

Overwriting climate_sift1.py


In [74]:
!python climate_sift1.py extract \
  --in_dir "./S-1_sections_extraction/chapters" \
  --out "./out_climate_text_filter/110325"  \
  --ann_total 0

[DONE] files=1315, paras=1579664, climate=21722


In [77]:
import pandas as pd

path = "/Users/panglinshao/Desktop/IPO/S-1/S-1 filings/out_climate_text_filter/110325/csv/paragraphs_pass_thresholds.csv"
df = pd.read_csv(path, dtype={"cik": "string"}) 
print("rows =", len(df))        

rows = 1579664
